# Fase 2 — Comparação de fontes de máscaras de referência

Diagnóstico comparativo das fontes candidatas de verdade de campo (MapBiomas e k-means AlphaEarth) para a classe café, na Região Geográfica Imediata de Guaxupé - MG (código IBGE 310044).

Este estágio carrega os GeoTIFF exportados na fase de aquisição, alinha as máscaras ao grid da composição Sentinel-2, binariza o mapa de clusters pela prevalência de café e produz a tabela de métricas pareadas (IoU, F1, precisão, recall e Kappa) em relação à referência MapBiomas.

## Detecção da raiz do repositório

Localiza a raiz do repositório a partir do diretório corrente e a insere no caminho de importação, garantindo o acesso ao pacote `src/`.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path


# Sobe os diretórios até encontrar src/config.yaml, marcador da raiz do projeto.
def _find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "src" / "config.yaml").is_file():
            return candidate
    raise RuntimeError("Raiz do repositório não localizada (src/config.yaml ausente).")


PROJECT_ROOT = _find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f"Raiz do projeto: {PROJECT_ROOT}")

## Detecção da plataforma

Identifica o ambiente de execução (Kaggle, Colab ou local) para adaptar a instalação de dependências e a leitura de segredos.

In [ ]:
import importlib.util
import os


def _is_colab_runtime() -> bool:
    """Detecta Colab com segurança, mesmo quando o pacote google não existe."""
    if "COLAB_GPU" in os.environ:
        return True
    try:
        return importlib.util.find_spec("google.colab") is not None
    except ModuleNotFoundError:
        return False


# Colab é detectado primeiro, pois /kaggle também existe nos runtimes do Colab.
def detect_platform() -> str:
    if _is_colab_runtime():
        return "colab"
    if "KAGGLE_KERNEL_RUN_TYPE" in os.environ or Path("/kaggle/working").is_dir():
        return "kaggle"
    return "local"


PLATFORM = detect_platform()
print(f"Plataforma detectada: {PLATFORM}")

## Instalação condicional das dependências

Em Kaggle/Colab instala o pacote com os extras geoespaciais e de aprendizado de máquina. No ambiente local a instalação é ignorada, pois é gerenciada por `uv` e pelo CI.

In [ ]:
import subprocess

# Instala o projeto editavelmente com os extras necessários apenas em nuvem.
if PLATFORM in {"kaggle", "colab"}:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-e", ".[geo,ml]"],
        cwd=PROJECT_ROOT,
        check=True,
    )
    print("Dependências instaladas.")
else:
    print("Ambiente local: instalação ignorada (gerenciada por uv/CI).")

## Carregamento da configuração única

Lê a configuração de `src/config.yaml` por meio de `src/config.py`, fonte única de verdade de caminhos, bandas, parâmetros e sementes.

In [ ]:
from src.config import CONFIG

# Exibe os parâmetros de máscaras definidos na configuração.
print(f"Fontes candidatas: {CONFIG.get('masks.candidates')}")
print(f"Fonte selecionada: {CONFIG.get('masks.selected_source')}")
print(f"Diretório local: {CONFIG.get('masks.local_dir')}")

## Fixação das sementes

Fixa as sementes de `python`, `numpy`, `torch` e `cuda` para garantir a reprodutibilidade das etapas amostrais (ex.: clusterização).

In [ ]:
from src.config import seed_everything

# Aplica a semente global definida na configuração.
resolved_seed = seed_everything()
print(f"Sementes fixadas em {resolved_seed}.")

## Carregamento de segredos

Em Kaggle/Colab os segredos são cadastrados no cofre da plataforma (Colab: painel Segredos na barra lateral; Kaggle: Add-ons → Secrets) e injetados nas variáveis de ambiente esperadas pelo pacote. A autenticação do Earth Engine aceita dois modos: conta de serviço (`GEE_SERVICE_ACCOUNT_EMAIL` + `GEE_SERVICE_ACCOUNT_KEY_JSON`) ou OAuth de usuário (`GEE_OAUTH_CREDENTIALS_JSON`), com `GEE_PROJECT` comum e obrigatório. Nenhum valor é impresso. No ambiente local, os segredos devem vir de variáveis de ambiente ou do arquivo `.env`.

In [ ]:
# Nomes das variáveis de ambiente consumidas pela aquisição.
GEE_COMMON_SECRETS = ("GEE_PROJECT",)
SERVICE_ACCOUNT_SECRETS = ("GEE_SERVICE_ACCOUNT_EMAIL", "GEE_SERVICE_ACCOUNT_KEY_JSON")
OAUTH_SECRETS = ("GEE_OAUTH_CREDENTIALS_JSON",)


def _load_secret(name: str) -> str:
    # Lê do cofre da plataforma (Colab ou Kaggle) e retorna o valor do segredo.
    if PLATFORM == "kaggle":
        from kaggle_secrets import UserSecretsClient

        return UserSecretsClient().get_secret(name)
    from google.colab import userdata

    return userdata.get(name)


if PLATFORM in {"kaggle", "colab"}:
    loaded: set[str] = set()
    for name in GEE_COMMON_SECRETS + SERVICE_ACCOUNT_SECRETS + OAUTH_SECRETS:
        try:
            os.environ[name] = _load_secret(name)
            loaded.add(name)
        except Exception:
            pass

    if "GEE_PROJECT" not in loaded:
        print("Segredo obrigatório ausente: GEE_PROJECT")
        print("Sem ele, a autenticação no Earth Engine falhará nas próximas células.")
    elif set(SERVICE_ACCOUNT_SECRETS).issubset(loaded):
        print("Autenticação GEE: conta de serviço (service account).")
    elif set(OAUTH_SECRETS).issubset(loaded):
        print("Autenticação GEE: OAuth de usuário.")
    else:
        print("Modo de autenticação GEE incompleto: cadastre a conta de serviço "
              "(GEE_SERVICE_ACCOUNT_EMAIL + GEE_SERVICE_ACCOUNT_KEY_JSON) ou o "
              "OAuth (GEE_OAUTH_CREDENTIALS_JSON).")
else:
    print("Ambiente local: segredos esperados via variáveis de ambiente/.env.")

## Autenticação no Google Earth Engine

Inicializa o Earth Engine com as credenciais lidas exclusivamente do ambiente: conta de serviço ou credenciais OAuth de usuário. Interrompe a execução caso as credenciais estejam ausentes, pois toda a fase depende do serviço.

In [ ]:
from src.data.gee_client import init_ee

# Inicializa o cliente do Earth Engine; sem credenciais a fase não prossegue.
ee = init_ee()
print("Earth Engine autenticado com sucesso.")

## Carregamento da área de estudo

Carrega a malha vetorial do IBGE e recorta o registro da Região Geográfica Imediata de Guaxupé, convertendo a geometria para o formato do Earth Engine.

In [ ]:
from src.data.aoi import geometry_bounds, geometry_to_ee, get_region_geometry

# Obtém a geometria unificada da região e a converte para ee.Geometry.
aoi_geometry = get_region_geometry()
aoi_ee = geometry_to_ee(ee, aoi_geometry)
print(f"Limites (minx, miny, maxx, maxy): {geometry_bounds(aoi_geometry)}")

## Resolução dos arquivos das máscaras

Deriva da configuração os nomes esperados dos GeoTIFF exportados no Drive e verifica a presença da composição e de cada fonte candidata no diretório local.

In [ ]:
from src.data.gee_client import make_export_description
from src.data.mask_compare import mask_candidate_filename

# Deriva caminhos esperados a partir da configuração (sem caminhos codificados).
prefix = CONFIG.get("gee.export_prefix")
region_code = CONFIG.get("aoi.region_code")
start_date = CONFIG.get("gee.start_date")
end_date = CONFIG.get("gee.end_date")
candidates = list(CONFIG.get("masks.candidates", []))
local_dir = CONFIG.paths.root / str(CONFIG.get("masks.local_dir"))
interim_dir = CONFIG.paths.interim / "masks"
interim_dir.mkdir(parents=True, exist_ok=True)

composite_file = local_dir / f"{make_export_description(prefix, region_code, start_date, end_date)}.tif"
candidate_files = {
    name: local_dir / mask_candidate_filename(name, prefix, region_code, start_date, end_date)
    for name in candidates
}
print(f"Composição: {composite_file.name} | presente={composite_file.is_file()}")
for name, path in candidate_files.items():
    print(f"{name}: {path.name} | presente={path.is_file()}")

## Carregamento do grid de referência

Lê uma banda da composição Sentinel-2 apenas para estabelecer o grid (dimensões e georreferenciamento) usado como referência no alinhamento das máscaras.

In [ ]:
from src.data.raster_io import read_band

# Lê uma banda da composição apenas para estabelecer o grid de referência.
reference_grid = read_band(composite_file, band=1)
print(f"Grid de referência (linhas, colunas): {reference_grid.shape}")

## Alinhamento das máscaras ao grid da composição

Reprojeta cada fonte candidata para o grid exato da composição Sentinel-2, garantindo o casamento pixel a pixel na comparação, e carrega os arranjos resultantes.

In [ ]:
import numpy as np

from src.data.raster_io import align_to_reference

# Reprojeta cada candidato para o grid exato da composição.
aligned: dict[str, np.ndarray] = {}
for name, source in candidate_files.items():
    aligned_file = interim_dir / f"{name}_aligned.tif"
    align_to_reference(source, composite_file, aligned_file)
    aligned[name] = read_band(aligned_file)
    print(f"{name}: {aligned[name].shape} | valores={np.unique(aligned[name])}")

## Binarização da máscara de clusters

Converte o mapa de clusters AlphaEarth em máscara binária, atribuindo café aos clusters cuja prevalência na referência MapBiomas supera o limiar configurado.

In [ ]:
from src.data.mask_compare import cluster_prevalence, threshold_clusters

# Converte o k-means em máscara binária pela prevalência de café por cluster.
threshold = float(CONFIG.get("masks.cluster_threshold", 0.5))
prevalence = cluster_prevalence(
    aligned["alphaearth_clusters"], aligned["mapbiomas_coffee"]
)
print(f"Prevalência por cluster: {prevalence}")
aligned["alphaearth_clusters"] = threshold_clusters(
    aligned["alphaearth_clusters"], prevalence, threshold
)
print(f"Valores binarizados: {np.unique(aligned['alphaearth_clusters'])}")

## Tabela comparativa de fontes

Calcula as métricas pareadas (IoU, F1, precisão, recall e Kappa) de cada fonte candidata em relação à referência MapBiomas e exibe o diagnóstico em tabela.

In [ ]:
from src.data.mask_compare import compare_sources

# Métricas pareadas em relação à referência MapBiomas.
report = compare_sources(aligned, reference_name="mapbiomas_coffee")
print(report.to_string(index=False))

## Visualização do grid comparativo

Renderiza a composição e as máscaras alinhadas em uma grade de inspeção visual, facilitando a conferência qualitativa das fontes de verdade de campo.

In [ ]:
import matplotlib.pyplot as plt

# Grade de inspeção visual: composição e máscaras alinhadas.
fig, axes = plt.subplots(1, len(aligned) + 1, figsize=(4 * (len(aligned) + 1), 4))
axes[0].imshow(reference_grid, cmap="gray")
axes[0].set_title("Sentinel-2 (banda de referência)")
axes[0].axis("off")
for ax, (name, mask) in zip(axes[1:], aligned.items()):
    ax.imshow(mask, cmap="gray", vmin=0, vmax=1)
    ax.set_title(name)
    ax.axis("off")
plt.tight_layout()
plt.show()